# Exploração CNES — estrutura de diálise, competência 2024-12

**Objetivo:** explorar o cadastro CNES de dezembro/2024 para descrever estabelecimentos com registro de diálise e sua distribuição territorial, sem tratar a contagem de estabelecimentos como capacidade efetivamente disponível.

O arquivo ZIP tem cerca de 2,67 GB descompactado. Este notebook lê apenas `tbDialise202412.csv`, `tbEstabelecimento202412.csv` e `tbMunicipio202412.csv` em streaming, sem extrair o pacote inteiro para disco ou carregar todos os registros ao mesmo tempo.


## Fonte e definição

Fonte local: `Dados/BASE_DE_DADOS_CNES_202412.ZIP`, base CNES da competência dezembro/2024. Referência para consulta: [CNES — serviços especializados](https://cnes2.datasus.gov.br/mod_ind_especialidades.asp).

A tabela `tbDialise` descreve campos de estrutura (salas e máquinas) associada ao estabelecimento. A existência de linha no cadastro não comprova funcionamento, atendimento SUS, vagas disponíveis ou produção. A análise aqui é exploratória e de competência única.


## 1. Localizar ZIP e inspecionar tabelas

In [5]:
from pathlib import Path
from zipfile import ZipFile
import csv, io

CANDIDATOS=[Path("../Dados/BASE_DE_DADOS_CNES_202412.ZIP"),Path("estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final/Dados/BASE_DE_DADOS_CNES_202412.ZIP")]
ZIP_PATH=next((p for p in CANDIDATOS if p.exists()),CANDIDATOS[0])
assert ZIP_PATH.exists(),f"ZIP CNES não encontrado: {ZIP_PATH}"
MEMBROS={
 "dialise":"tbDialise202412.csv",
 "estabelecimento":"tbEstabelecimento202412.csv",
 "municipio":"tbMunicipio202412.csv",
}
with ZipFile(ZIP_PATH) as z:
    disponiveis=set(z.namelist())
    for apelido,nome in MEMBROS.items():
        assert nome in disponiveis, f"Membro ausente no ZIP: {nome}"
        print(apelido,nome,f"{z.getinfo(nome).file_size/1_000_000:.1f} MB descompactado")


dialise tbDialise202412.csv 0.2 MB descompactado
estabelecimento tbEstabelecimento202412.csv 260.4 MB descompactado
municipio tbMunicipio202412.csv 0.4 MB descompactado


## 2. Leitura em streaming de registros selecionados

In [6]:
def iterar_csv_cnes(nome_membro):
    """Lê linhas latin-1 separadas por ponto e vírgula diretamente do ZIP."""
    with ZipFile(ZIP_PATH) as z:
        with z.open(nome_membro) as binario:
            texto=io.TextIOWrapper(binario,encoding="latin-1",newline="")
            leitor=csv.DictReader(texto,delimiter=";")
            for linha in leitor: yield linha

# Indexa apenas registros de diálise e os metadados básicos de estabelecimento/município.
dialise=list(iterar_csv_cnes(MEMBROS["dialise"]))
assert dialise, "Tabela tbDialise sem registros."
print("Registros tbDialise:",len(dialise))
print("Campos:",list(dialise[0].keys()))

estab={r["CO_UNIDADE"]:r for r in iterar_csv_cnes(MEMBROS["estabelecimento"])}
municipios={r["CO_MUNICIPIO"]:r for r in iterar_csv_cnes(MEMBROS["municipio"])}
print("Estabelecimentos indexados:",len(estab))
print("Municípios indexados:",len(municipios))
print("Campos município:",list(next(iter(municipios.values())).keys()))


Registros tbDialise: 1258
Campos: ['CO_UNIDADE', 'QT_SALA_HBSAG_POS', 'QT_SALA_HBSAG_NEG', 'QT_SALA_DPI', 'QT_SALA_DPAC', 'QT_SALA_REAG_POS', 'QT_SALA_REAG_NEG', 'QT_SALA_REHCV', 'NU_MAQH_PROP', 'NU_MAQH_OUTR', 'CO_NEFRO_RESPONSAVEL', 'CO_CPF_NEFRO', 'CO_CPF_DIRETOR', 'NO_DIRETOR_RESPONSAVEL', 'TP_FILTRO_AREIA', 'TP_FILTRO_CARVAO', 'TP_ABRANDADOR', 'TP_DEOINIZADOR', 'TP_OSMOSE_REVERSA', 'TP_OUTROS_TRAT_AGUA', "TO_CHAR(DT_ATUALIZACAO,'DD/MM/YYYY')", 'CO_USUARIO', "TO_CHAR(DT_ATUALIZACAO_ORIGEM,'DD/MM/YYYY')"]
Estabelecimentos indexados: 557329
Municípios indexados: 5606
Campos município: ['CO_MUNICIPIO', 'NO_MUNICIPIO', 'CO_SIGLA_ESTADO', 'TP_CADASTRO', 'TP_PACTO', 'TP_ENVIA', 'TP_ENVIA_CNES', 'TP_CIB_SAS', 'TP_PLENO_ORIGEM', 'TP_MAC', 'NU_POPULACAO', 'NU_DENSIDADE', 'CMTP_INICIO_MAC']


## 3. Verificar chaves e montar tabela analítica

In [7]:
chaves=[r["CO_UNIDADE"] for r in dialise]
assert len(chaves)==len(set(chaves)), "CO_UNIDADE repetido em tbDialise."
sem_estab=[c for c in chaves if c not in estab]
assert not sem_estab, f"Unidades de diálise sem correspondência em tbEstabelecimento: {len(sem_estab)}"

# Campos numéricos de estrutura: vazio continua ausente, não é convertido em zero.
campos_qt=["QT_SALA_HBSAG_POS","QT_SALA_HBSAG_NEG","QT_SALA_DPI","QT_SALA_DPAC","QT_SALA_REAG_POS","QT_SALA_REAG_NEG","QT_SALA_REHCV","NU_MAQH_PROP","NU_MAQH_OUTR"]
def numero_ou_none(x):
    x=(x or "").strip()
    return int(x) if x else None

analitica=[]
for d in dialise:
    e=estab[d["CO_UNIDADE"]]
    cod_mun=e.get("CO_MUNICIPIO_GESTOR","").strip()
    m=municipios.get(cod_mun,{})
    rec={
        "co_unidade":d["CO_UNIDADE"],
        "cnes":e.get("CO_CNES",""),
        "nome":e.get("NO_FANTASIA","").strip(),
        "cod_municipio":cod_mun,
        "municipio":m.get("NO_MUNICIPIO",""),
        "cod_uf":m.get("CO_SIGLA_ESTADO",""),
        "uf":m.get("CO_SIGLA_ESTADO",""),
        "atende_sus":None,  # atendimento SUS ainda não filtrado
    }
    for campo in campos_qt: rec[campo]=numero_ou_none(d.get(campo))
    analitica.append(rec)

print("Registros ligados a estabelecimento e município:",len(analitica))
print("Sem UF identificada:",sum(not r["uf"] for r in analitica))


Registros ligados a estabelecimento e município: 1258
Sem UF identificada: 0


## 4. Resumos descritivos de estrutura cadastrada

In [8]:
por_uf={}
for r in analitica:
    uf=r["uf"] or "UF não identificada"
    por_uf[uf]=por_uf.get(uf,0)+1
print("Estabelecimentos com registro de diálise por UF:")
for uf,n in sorted(por_uf.items(),key=lambda x:x[1],reverse=True): print(uf,n)

for campo in ["NU_MAQH_PROP","NU_MAQH_OUTR"]:
    conhecidos=[r[campo] for r in analitica if r[campo] is not None]
    print(f"\n{campo}: registros não ausentes={len(conhecidos)}, soma={sum(conhecidos):,}, mediana={sorted(conhecidos)[len(conhecidos)//2] if conhecidos else None}".replace(",","."))

# Atalho de triagem: detectar linhas com pelo menos uma máquina cadastrada.
com_maquina=sum((r["NU_MAQH_PROP"] or 0)+(r["NU_MAQH_OUTR"] or 0)>0 for r in analitica)
print("Unidades com pelo menos uma máquina registrada:",com_maquina,"de",len(analitica))


Estabelecimentos com registro de diálise por UF:
SP 283
RJ 130
MG 128
RS 109
BA 60
PR 58
SC 48
DF 47
GO 42
PA 41
CE 37
PE 36
MA 28
ES 26
PB 24
PI 24
MT 23
MS 19
RN 19
RO 18
AL 17
AM 10
SE 8
TO 8
AC 6
RR 6
AP 3

NU_MAQH_PROP: registros não ausentes=1256. soma=32.431. mediana=23

NU_MAQH_OUTR: registros não ausentes=1256. soma=3.790. mediana=0
Unidades com pelo menos uma máquina registrada: 1180 de 1258


### Interpretação, limitações e próximos passos

- A contagem representa registros da competência 2024-12 em `tbDialise`, ligados às tabelas cadastrais. Não é série temporal nem contagem de pacientes ou capacidade assistencial efetiva.
- A extração atual não filtra atendimento SUS. Antes de chamar o resultado de “oferta SUS”, é preciso aplicar campos apropriados de atendimento SUS e serviço/classificação, conferir dicionário de dados e documentar a regra.
- Campos de sala/máquina indicam estrutura informada no cadastro; não equivalem a turnos, vagas, disponibilidade operacional ou produção.
- Verificar códigos geográficos e chaves com a documentação CNES da competência antes de publicar agregações.
- Próximo passo recomendado: validar a regra do serviço/classificação para diálise e atendimento SUS (tabelas `rlEstabServClass`, `tbServicoEspecializado` e `tbClassificacaoServico`) e então comparar estabelecimentos ativos por UF.
